# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (page), evaluated at a decision date.**
For a given `content_hash_id`, I aggregate its daily performance from
`fact_content_daily_performance` into two adjacent 30-day windows that
end at the decision date, then compare them.

**Table(s) used:** `fact_content_daily_performance` (daily grain, the
source of the label) joined to `dim_content` (static page metadata,
the source of features).

**Time window:** decision date = `2026-03-31`.
- current window: `2026-03-02` → `2026-03-31` (last 30 days)
- prior window: `2026-01-31` → `2026-03-01` (the 30 days before that)
Both windows sit inside the mid-panel month `2026-03` I'm iterating on
— the sealed `_sample` table (June 2026) is never touched here.

**Label / proxy:** `is_declining_label` = 1 if impressions in the
current window dropped more than 20% vs. the prior window, else 0.
Same real, observed-outcome definition as my CSV work — now computed
by hand from raw daily rows instead of read pre-built.

**Deliberately excluded:** any column computed from the current window
itself (e.g. current-window impressions) — using it as a feature would
leak the answer. Also excluded: `provider_used`/`model_used` (flagged
not-a-feature in the data dictionary), and any FlyRank product flags —
those are outputs of an existing rule, not inputs to mine.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

import duckdb
con = duckdb.connect()

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN: ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIMCL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

print(con.sql(f"SELECT COUNT(*) n, MIN(report_date) lo, MAX(report_date) hi FROM {FACT}").df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

          n         lo         hi
0  78835655 2025-01-27 2026-06-30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` (prior 30d, summed) | Feature | Knowable before the current window opens |
| `gsc_avg_position` (prior 30d, avg) | Feature | Same — historical, not from the label window |
| `content_age_days` (derived: decision_date − `content_created_date`) | Feature | Fixed fact as of decision date, always knowable |
| `word_count` (dim_content) | Feature | Content property, doesn't change with traffic |
| `content_type` (dim_content) | Feature | Content property, set at publish time |
| `gsc_impressions` (current 30d) | Label input | Used ONLY to compute `is_declining_label`, never a feature |
| `is_declining_label` | Label | The target |
| `content_hash_id`, `client_hash_id` | Context | Joining/grouping only, never fed to a model |
| `keyword_hash_id`, `url_hash_id` | Context | Identifiers, joining only |
| `provider_used`, `model_used` | Excluded | Data dictionary flags these as not-a-feature |
| `is_published`, `is_deleted` | Excluded (filter only) | Used to keep the slice to live, non-deleted pages — not fed to the model |

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims from Section 1, each checked with a query on `month=2026-03`.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print("Rows violating the grain (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain (should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []


In [13]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) n_rows,
           COUNT(DISTINCT content_hash_id) n_pages,
           MIN(report_date) lo, MAX(report_date) hi
    FROM {FACT}
    WHERE report_date >= DATE '2026-01-31' AND report_date <= DATE '2026-03-31'
""").df()
print(slice_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     n_rows  n_pages         lo         hi
0  17456920   349411 2026-01-31 2026-03-31


In [14]:
before = con.sql(f"""
    SELECT COUNT(*) n
    FROM {FACT}
    WHERE report_date >= DATE '2026-01-31' AND report_date <= DATE '2026-03-31'
""").df()

after = con.sql(f"""
    SELECT COUNT(*) n
    FROM {FACT}
    WHERE report_date >= DATE '2026-01-31' AND report_date <= DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

print("Before filter:", before["n"][0])
print("After IS TRUE filter:", after["n"][0])
print(f"Survival rate: {after['n'][0]/before['n'][0]:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Before filter: 17456920
After IS TRUE filter: 492319
Survival rate: 2.8%


1. `impressions_prior30` — only aggregated over the window ending before the current (label) window starts.
2. `avg_position_prior30` — same reasoning, purely historical.
3. `content_age_days` — computed as decision_date minus `content_created_date`; a fixed fact, no future info.
4. `word_count` — static content property from `dim_content`.
5. `content_type` — static content property, set at publish time.

In [15]:
DECISION_DATE = "2026-03-31"

feature_frame = con.sql(f"""
    WITH prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30,
               AVG(gsc_avg_position) AS avg_position_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    current AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_current30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT p.client_hash_id, p.content_hash_id,
           p.impressions_prior30, p.avg_position_prior30,
           c.impressions_current30,
           CASE WHEN c.impressions_current30 < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM prior p
    JOIN current c USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

feats = con.sql(f"""
    SELECT content_hash_id, content_type, word_count,
           DATE_DIFF('day', content_created_date, DATE '{DECISION_DATE}') AS content_age_days
    FROM {DIMC}
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

feature_frame = feature_frame.merge(feats, on="content_hash_id", how="left")
print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(135113, 9)


,client_hash_id,content_hash_id,impressions_prior30,avg_position_prior30,impressions_current30,is_declining_label,content_type,word_count,content_age_days
0,client_e547b89c05043229,content_6cebaece2c138a05,1703.0,8.001150,7719.0,0,keyword article,2925,351.0
1,client_e547b89c05043229,content_72a4cc98359e86d1,313.0,3.604267,354.0,0,keyword article,2736,351.0
2,client_e547b89c05043229,content_5e24b85d613aef32,3761.0,5.440261,4284.0,0,keyword article,3184,351.0
3,client_e547b89c05043229,content_bfed28eb035db5b2,600.0,5.372035,535.0,0,keyword article,2926,351.0
4,client_e547b89c05043229,content_a59f8c5fa4bfa799,1485.0,4.355244,3955.0,0,keyword article,3162,351.0


**Deliberate leak:** `impressions_current30` is literally the number
`is_declining_label` is computed from. Adding it as a "feature" should
make a toy model look almost perfect — that's the tell that something's
wrong, not that the model is good.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

honest_cols = ["impressions_prior30", "avg_position_prior30", "content_age_days", "word_count"]

X = feature_frame[honest_cols].fillna(0)
y = feature_frame["is_declining_label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_score = honest_model.score(Xte, yte)
print(f"Honest accuracy (4 real features): {honest_score:.3f}")

leaky_cols = honest_cols + ["impressions_current30"]
Xl = feature_frame[leaky_cols].fillna(0)
Xltr, Xlte, yltr, ylte = train_test_split(Xl, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xltr, yltr)
leaky_score = leaky_model.score(Xlte, ylte)
print(f"Leaky accuracy (+ current30 impressions): {leaky_score:.3f}")

print(f"\nJump from leak: {honest_score:.3f} -> {leaky_score:.3f}")
print("Leak column removed. Keeping the honest number:", round(honest_score, 3))


Honest accuracy (4 real features): 0.760
Leaky accuracy (+ current30 impressions): 1.000

Jump from leak: 0.760 -> 1.000
Leak column removed. Keeping the honest number: 0.76


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


**This data can never tell me *why* a page declined** — only that
impressions dropped between two windows. A drop could mean the content
got worse, a competitor outranked it, or Google changed the SERP layout
for that query — the warehouse has no signal to tell those apart.

**GA4 coverage is far sparser than GSC coverage.** Requiring both
`gsc_data_available IS TRUE` and `ga4_data_available IS TRUE` on the
same 60-day slice drops survival from 17,456,920 rows to 492,319 —
just 2.8%. My label and features only depend on GSC impressions, so I
don't apply the GA4 filter in my actual pipeline, but the gap is worth
flagging: any future feature that pulls in GA4 metrics (pageviews,
sessions, engagement) would be working with a tiny, likely non-random
slice of pages, not the full inventory.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.